In 1.0.1.A and 1.1.1.A we met random search and gradient descent. These are essential concepts for understanding machine learning: how do we define a loss function for a problem and optimize for the minimum loss? This is all you *need* to get started with machine learning, but the genetic algorithm is an interesting and related concept so this notebook presents that. Really optimization could be an entire course in itself, but we need to move on eventually. 

The genetic algorithm uses an analogy to natural selection: we generate a population of simulated individuals whose "genes" represent a solution to a problem. Then we have them "reproduce" and "mutate" to get related individuals with similar but distinct solutions to the problem, and we have a "population" that the aagents compete to be in. This is especially good for discrete domain problems, where we can't do gradient descent (which requires a continuous, differentiable domain) and we want to do better than guessing randomly.

### Imports & Seeding

In [1]:
import random as r
r.seed(1)

# Travelling Salesman Problem

A classic problem in computer science is this: we have N cities spread randomly and we have to find the best order in which to visit the cities such that we start in one city then visit all the cities once and return to the start, minimizing the distance travelled. This is a discrete domain problem because our solution would look something like: {4, 3, 1, 2, 5}. Notice that to check every solution exhaustively and naively is $O(n!)$ because there are $n!$ ways to arrange $n$ things. Of course the space is actually smaller, since for example ${4, 3, 1, 2, 5}$ is effectively the same solution as ${1, 2, 5, 4, 3}$ since we visit the cities in the same order and where we start doesn't matter for the total distance travelled. That said, exhaustively searching every combination is going to be computationally infeasible for large $n$, hence the need for the genetic algorithm!

In [2]:
num_cities = 28
num_individuals = 50
num_iterations = 1000
def generate_cities(N):
    return([[r.uniform(-1, 1), r.uniform(-1, 1)] for n in range(N)])
cities = generate_cities(num_cities)

def salesman_loss(some_solution):
    output = 0
    for i in range(len(some_solution) - 1):
        output += (some_solution[i][0] - some_solution[i+1][0]) ** 2  + (some_solution[i][1] - some_solution[i+1][1]) ** 2 
    #Go back to the first city
    output += (some_solution[-1][0] - some_solution[0][0]) ** 2  + (some_solution[-1][1] - some_solution[0][1]) ** 2 
    return(output)

# Genetic Algorithm

So our algorithm works like this. Generate $P$ individuals, composed of a "genome" (solution to TSP) and a "fitness" (negative loss). For $N$ iterations we generate a new individual, compare it to the population in turn, if it is fitter than at least one individual we replace them, and if not we discard the new individual. The generation could be sexual (pick parent individuals and combine them in some way) or asexual (just pick an individual at random). Then  we "mutate" the individual by allowing it to vary slightly: here that looks like randomly swapping some genes. 

In [3]:
#Turn order into loss
def genetic_fitness(solution, cities = cities):
    order = [cities[i] for i in solution]
    return(-salesman_loss(order))

#For the initial population
def generate_individual():
    genes = [c for c in range(num_cities)]
    r.shuffle(genes)
    return([genes, genetic_fitness(genes)])

#Note that this doesn't produce exactly "swaps" swaps as there is some possibility m = n
def mutate_genome(genome, mutation_rate = 0.1):
    swaps = 0
    for i in range(num_cities):
        if r.uniform(0, 1) < mutation_rate:
            swaps += 1
    output_genome = [g for g in genome]
    for j in range(swaps):
        m = r.randint(0, num_cities - 1)
        n = r.randint(0, num_cities - 1)
        output_genome[m], output_genome[n] = output_genome[n], output_genome[m]
    return(output_genome)

def breed(population):
    to_reproduce = r.choice(population)
    genome = mutate_genome(to_reproduce[0])
    offspring = [genome, genetic_fitness(genome)]
    stopping_condition = False
    i = 0
    while not stopping_condition:
        if offspring[1] > population[i][1]:
            population[i] = offspring
            stopping_condition = True
        if i == num_cities - 1:
            stopping_condition = True
        i += 1
    return(population)

population = [generate_individual() for n in range(num_individuals)]
for n in range(num_iterations):
    population = breed(population)
print(f"solution: {population[0]}")

solution: [[24, 23, 13, 25, 22, 20, 27, 0, 8, 26, 5, 1, 19, 7, 3, 11, 9, 17, 2, 12, 16, 15, 10, 4, 21, 14, 6, 18], -21.64947123945289]


# Task: Sexual Genetic Algorithm & Knapsack Problem

Earlier we saw the *asexual genetic algorithm*. For travelling salesman it is convenient to have the individuals reproduce asexually because there isn't a convenient way to have them exchange genomes. For your task, investigate the *knapsack problem* and implement a version of the sexual genetic algorithm.

The knapsack problem is this: suppose we have $N$ items and a knapsack with a weight limit of $W$ and must decide which items to take with us given each item has a utility $u$ and a weight $w$. If our chosen items weigh less than $W$ in total our utility is zero. If our chosen items weigh less than $W$ in total, our utility is the sum of each item's $u$.

Solutions will look something like: {0, 1, 1, 1, 0, 0} i.e. in this case we take items 1, 2, and 3 but not 0, 4, or 5.

Implement the *sexual genetic algorithm* to solve the *knapsack problem*. It works like the asexual genetic algorithm only we pick two (or perhaps more? We're not limited by biology here!) parent solutions anff combine them in some way.